In [1]:
from langchain_community.document_loaders import PyPDFLoader

In [4]:
loader  = PyPDFLoader("jesc101.pdf")

docs = loader.load()
print(docs[0].page_content)

Chemical Reactions
and Equations
1CHAPTER
C
onsider the following situations of daily life and think what happens
when –
/square6milk is left at room temperature during summers.
/square6an iron tawa/pan/nail is left exposed to humid atmosphere.
/square6grapes get fermented.
/square6food is cooked.
/square6food gets digested in our body.
/square6we respire.
In all the above situations, the nature and the identity of the initial
substance have somewhat changed. We have already learnt about physical
and chemical changes of matter in our previous classes. Whenever a chemical
change occurs, we can say that a chemical reaction has taken place.
You may perhaps be wondering as to what is actually meant by a
chemical reaction. How do we come to know that a chemical reaction
has taken place? Let us perform some activities to find the answer to
these questions.
Figure 1.1
Burning of a magnesium ribbon in air and collection of magnesium
oxide in a watch-glass
Activity 1.1Activity 1.1Activity 1.1Ac

In [5]:
meta  = {
    "source": "jesc101.pdf",
    "title": "class 10th Science",
    "Board": "CBSE",
    "Language": "English",
    "Topic": "Science",
    "chapter": "1",
    "chapter_name": "Chemical Reactions and Equations"
}

In [6]:
meta

{'source': 'jesc101.pdf',
 'title': 'class 10th Science',
 'Board': 'CBSE',
 'Language': 'English',
 'Topic': 'Science',
 'chapter': '1',
 'chapter_name': 'Chemical Reactions and Equations'}

## chunking 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, Language
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker

In [26]:
for doc in docs:
    doc.metadata.pop("source","producer")

print(docs[0].metadata)

{'producer': 'GPL Ghostscript 8.15', 'creator': 'PageMaker 7.0', 'creationdate': '2017-12-20T12:15:25+00:00', 'author': 'dtpcell5', 'moddate': '2026-03-24T12:00:47+05:30', 'title': 'CHAP 1.pmd', 'total_pages': 16, 'page': 0, 'page_label': '1'}


In [28]:
for doc in docs:
    doc.metadata.pop("creator", "title")

print(docs[0].metadata) 

{'producer': 'GPL Ghostscript 8.15', 'creationdate': '2017-12-20T12:15:25+00:00', 'author': 'dtpcell5', 'moddate': '2026-03-24T12:00:47+05:30', 'title': 'CHAP 1.pmd', 'total_pages': 16, 'page': 0, 'page_label': '1'}


In [29]:
for doc in docs:
    doc.metadata = meta

print(docs[0].metadata)

{'source': 'jesc101.pdf', 'title': 'class 10th Science', 'Board': 'CBSE', 'Language': 'English', 'Topic': 'Science', 'chapter': '1', 'chapter_name': 'Chemical Reactions and Equations'}


In [59]:
# 1 length based chunking
splitter = CharacterTextSplitter( 
    chunk_size=500,
    chunk_overlap=50,
    separator="\n\n"
    )

result = splitter.split_documents(docs)
print(result[0].metadata)


{'source': 'jesc101.pdf', 'title': 'class 10th Science', 'Board': 'CBSE', 'Language': 'English', 'Topic': 'Science', 'chapter': '1', 'chapter_name': 'Chemical Reactions and Equations'}


In [60]:
# 2. Structure based chunking

splitter2 = RecursiveCharacterTextSplitter(
    chunk_size = 900,
    chunk_overlap = 50
)
result1 = splitter2.split_documents(docs)
print(result1[0])

page_content='Chemical Reactions
and Equations
1CHAPTER
C
onsider the following situations of daily life and think what happens
when –
/square6milk is left at room temperature during summers.
/square6an iron tawa/pan/nail is left exposed to humid atmosphere.
/square6grapes get fermented.
/square6food is cooked.
/square6food gets digested in our body.
/square6we respire.
In all the above situations, the nature and the identity of the initial
substance have somewhat changed. We have already learnt about physical
and chemical changes of matter in our previous classes. Whenever a chemical
change occurs, we can say that a chemical reaction has taken place.
You may perhaps be wondering as to what is actually meant by a
chemical reaction. How do we come to know that a chemical reaction
has taken place? Let us perform some activities to find the answer to
these questions.
Figure 1.1' metadata={'source': 'jesc101.pdf', 'title': 'class 10th Science', 'Board': 'CBSE', 'Language': 'English', 'Topi

In [61]:
# 3. Sematic Chunking

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6607.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [66]:
splitter3 = SemanticChunker(
    embeddings = embeddings,
    breakpoint_threshold_type= "percentile",
    breakpoint_threshold_amount=90
)

In [67]:
result3 = splitter3.create_documents([doc.page_content for doc in docs], metadatas=[doc.metadata for doc in docs])

In [68]:
print(result3)

[Document(metadata={'source': 'jesc101.pdf', 'title': 'class 10th Science', 'Board': 'CBSE', 'Language': 'English', 'Topic': 'Science', 'chapter': '1', 'chapter_name': 'Chemical Reactions and Equations'}, page_content='Chemical Reactions\nand Equations\n1CHAPTER\nC\nonsider the following situations of daily life and think what happens\nwhen –\n/square6milk is left at room temperature during summers. /square6an iron tawa/pan/nail is left exposed to humid atmosphere. /square6grapes get fermented. /square6food is cooked. /square6food gets digested in our body. /square6we respire. In all the above situations, the nature and the identity of the initial\nsubstance have somewhat changed. We have already learnt about physical\nand chemical changes of matter in our previous classes. Whenever a chemical\nchange occurs, we can say that a chemical reaction has taken place. You may perhaps be wondering as to what is actually meant by a\nchemical reaction. How do we come to know that a chemical reac

In [69]:
print("=" * 50)
print("CHUNKING COMPARISON SUMMARY")
print("=" * 50)

for name, chunks in [
    ("Character Based", result),
    ("Structure Based", result1),
    ("Semantic Based",  result3)
]:
    avg_size = sum(len(c.page_content) for c in chunks) // len(chunks)
    min_size = min(len(c.page_content) for c in chunks)
    max_size = max(len(c.page_content) for c in chunks)
    
    print(f"\n{name}")
    print(f"  Total chunks : {len(chunks)}")
    print(f"  Avg size     : {avg_size} chars")
    print(f"  Min size     : {min_size} chars")
    print(f"  Max size     : {max_size} chars")

CHUNKING COMPARISON SUMMARY

Character Based
  Total chunks : 16
  Avg size     : 2000 chars
  Min size     : 1565 chars
  Max size     : 2781 chars

Structure Based
  Total chunks : 44
  Avg size     : 737 chars
  Min size     : 164 chars
  Max size     : 900 chars

Semantic Based
  Total chunks : 58
  Avg size     : 551 chars
  Min size     : 5 chars
  Max size     : 1714 chars


### Embeddings

In [ ]:
store_char = Chroma.from_documents(
    documents=result,
    embedding=embeddings,
    collection_name="character_based",
    persist_directory="./chroma_db/character"
)

store_struct = Chroma.from_documents(
    documents=result1,
    embedding=embeddings,
    collection_name="structure_based",
    persist_directory="./chroma_db/structure"
)

store_semantic = Chroma.from_documents(
    documents=result3,
    embedding=embeddings,
    collection_name="semantic_based",
    persist_directory="./chroma_db/semantic"
)

In [98]:
# search 
vector_store.similarity_search(
    query = "what is chemical reaction?",
    k=1
)


[Document(metadata={'Board': 'CBSE', 'source': 'jesc101.pdf', 'Topic': 'Science', 'title': 'class 10th Science', 'chapter': '1', 'chapter_name': 'Chemical Reactions and Equations', 'Language': 'English'}, page_content='Chemical Reactions\nand Equations\n1CHAPTER\nC\nonsider the following situations of daily life and think what happens\nwhen –\n/square6milk is left at room temperature during summers. /square6an iron tawa/pan/nail is left exposed to humid atmosphere. /square6grapes get fermented. /square6food is cooked. /square6food gets digested in our body. /square6we respire. In all the above situations, the nature and the identity of the initial\nsubstance have somewhat changed. We have already learnt about physical\nand chemical changes of matter in our previous classes. Whenever a chemical\nchange occurs, we can say that a chemical reaction has taken place. You may perhaps be wondering as to what is actually meant by a\nchemical reaction. How do we come to know that a chemical reac

In [ ]:
# method 2
retriever = vector_store.as_retriever(search_kwargs={"k":1})
query = "what is hit-and-trial method"
results = retriever.invoke(query)

In [115]:
for i, doc in enumerate(results):
    print(f"\nResult {i+1}:")
    print(doc.page_content)


Result 1:
Chemical Reactions and Equations 5
To equalise Fe, we take three atoms of Fe on the LHS. 3   Fe   +   4   H2O   →   Fe3O4   +   4   H2 (1.8)
Step VI: Finally, to check the correctness of the balanced equation, we
count atoms of each element on both sides of the equation. 3Fe + 4H2O → Fe3O4 + 4H2
The numbers of atoms of elements on both sides of Eq. (1.9) are
equal. This equation is now balanced. This method of balancing chemical
equations is called hit-and-trial method as we make trials to balance
the equation by using the smallest whole number coefficient. Step VII: Writing Symbols of Physical StatesWriting Symbols of Physical StatesWriting Symbols of Physical StatesWriting Symbols of Physical StatesWriting Symbols of Physical States   Car efully examine
the above balanced Eq.
